# Agentic RAG System - Google Colab

This notebook implements an agentic RAG system with 10+ LLM calls per query and tracks detailed metrics.

## Features:
- 10 specialized agents
- Complete tracking of tokens, costs, and latency
- JSON output with aggregated statistics
- ChromaDB integration for RAG
- MCP Servers for GitHub and StackOverflow

## Install Dependencies

In [ ]:
!pip install -q openai chromadb requests beautifulsoup4 python-dotenv tqdm

## Configure API Key

In [ ]:
import os
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in the sidebar and add OPENAI_API_KEY
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("API Key loaded from Colab Secrets")
except:
    # Option 2: Enter manually (less secure)
    OPENAI_API_KEY = input("Enter your OpenAI API Key: ")
    print("API Key entered manually")

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# Optional: GitHub Token for MCP Server
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
    print("GitHub Token loaded")
except:
    print("GitHub Token not found (optional)")

## Agent Definitions

In [ ]:
from openai import OpenAI
import json

class IntentClassifier:
    def __init__(self, client):
        self.client = client
    
    def classify(self, query: str) -> dict:
        response = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": "Classify the user query into: conceptual, how_to, troubleshooting, or comparison. Return JSON with 'category' and 'confidence'."
            }, {
                "role": "user",
                "content": query
            }],
            response_format={"type": "json_object"}
        )
        
        result = json.loads(response.choices[0].message.content)
        return {
            'category': result.get('category', 'conceptual'),
            'confidence': result.get('confidence', 0.8),
            'tokens': {
                'input': response.usage.prompt_tokens,
                'output': response.usage.completion_tokens
            }
        }

class QueryRewriter:
    def __init__(self, client):
        self.client = client
    
    def rewrite(self, query: str, intent: str) -> dict:
        response = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": f"Rewrite the query for better RAG retrieval. Intent: {intent}. Return JSON with 'rewritten_query' and 'keywords' array."
            }, {
                "role": "user",
                "content": query
            }],
            response_format={"type": "json_object"}
        )
        
        result = json.loads(response.choices[0].message.content)
        return {
            'rewritten_query': result.get('rewritten_query', query),
            'keywords': result.get('keywords', []),
            'tokens': {
                'input': response.usage.prompt_tokens,
                'output': response.usage.completion_tokens
            }
        }

class RAGScorer:
    def __init__(self, client):
        self.client = client
    
    def score_chunks(self, query: str, chunks: list) -> dict:
        chunks_text = "\n\n".join([f"Chunk {i+1}: {c['text'][:200]}..." for i, c in enumerate(chunks)])
        
        response = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": "Score each chunk by relevance to the query (0-10). Return JSON with 'scores' as an array of numbers, one score per chunk in order. Example: {\"scores\": [8, 6, 9, 7, 5]}"
            }, {
                "role": "user",
                "content": f"Query: {query}\n\nChunks:\n{chunks_text}"
            }],
            response_format={"type": "json_object"}
        )
        
        result = json.loads(response.choices[0].message.content)
        scores_raw = result.get('scores', [10] * len(chunks))
        
        # Ensure scores are numeric and match chunk count
        scores = []
        for i, score in enumerate(scores_raw[:len(chunks)]):
            try:
                # Convert to float if it's a dict or other type
                if isinstance(score, dict):
                    scores.append(float(score.get('score', 5)))
                else:
                    scores.append(float(score))
            except (ValueError, TypeError):
                scores.append(5.0)
        
        # Pad with default scores if needed
        while len(scores) < len(chunks):
            scores.append(5.0)
        
        # Sort by score
        chunk_score_pairs = list(zip(chunks, scores))
        ranked = sorted(chunk_score_pairs, key=lambda x: x[1], reverse=True)
        
        return {
            'ranked_chunks': [c for c, s in ranked],
            'scores': [s for c, s in ranked],
            'tokens': {
                'input': response.usage.prompt_tokens,
                'output': response.usage.completion_tokens
            }
        }

class ContextSynthesizer:
    def __init__(self, client):
        self.client = client
    
    def synthesize(self, query: str, chunks: list, github_issues: list, so_posts: list) -> dict:
        context = "\n\n".join([c['text'] for c in chunks[:3]])
        
        response = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": "Synthesize context from multiple sources into a coherent summary."
            }, {
                "role": "user",
                "content": f"Query: {query}\n\nContext: {context}"
            }]
        )
        
        return {
            'synthesized_context': response.choices[0].message.content,
            'sources_used': len(chunks) + len(github_issues) + len(so_posts),
            'tokens': {
                'input': response.usage.prompt_tokens,
                'output': response.usage.completion_tokens
            }
        }

class ResponseGenerator:
    def __init__(self, client):
        self.client = client
    
    def generate(self, query: str, context: str, intent: str) -> dict:
        response = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": f"Generate a comprehensive answer based on the context. Intent: {intent}"
            }, {
                "role": "user",
                "content": f"Query: {query}\n\nContext: {context}"
            }]
        )
        
        answer = response.choices[0].message.content
        has_code = '```' in answer
        
        return {
            'answer': answer,
            'has_code': has_code,
            'tokens': {
                'input': response.usage.prompt_tokens,
                'output': response.usage.completion_tokens
            }
        }

class CodeValidator:
    def __init__(self, client):
        self.client = client
    
    def validate(self, response: str, context: str) -> dict:
        result = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": "Validate code examples for correctness. Return JSON with 'is_valid' boolean."
            }, {
                "role": "user",
                "content": f"Response: {response}\n\nContext: {context}"
            }],
            response_format={"type": "json_object"}
        )
        
        validation = json.loads(result.choices[0].message.content)
        return {
            'is_valid': validation.get('is_valid', True),
            'tokens': {
                'input': result.usage.prompt_tokens,
                'output': result.usage.completion_tokens
            }
        }

class QualityEvaluator:
    def __init__(self, client):
        self.client = client
    
    def evaluate(self, query: str, response: str) -> dict:
        result = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": "Evaluate response quality (0-10). Return JSON with 'scores' object containing 'overall'."
            }, {
                "role": "user",
                "content": f"Query: {query}\n\nResponse: {response}"
            }],
            response_format={"type": "json_object"}
        )
        
        evaluation = json.loads(result.choices[0].message.content)
        return {
            'scores': evaluation.get('scores', {'overall': 8}),
            'tokens': {
                'input': result.usage.prompt_tokens,
                'output': result.usage.completion_tokens
            }
        }

class HallucinationDetector:
    def __init__(self, client):
        self.client = client
    
    def detect(self, response: str, context: str) -> dict:
        result = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": "Detect hallucinations. Return JSON with 'is_grounded' boolean and 'verdict'."
            }, {
                "role": "user",
                "content": f"Response: {response}\n\nContext: {context}"
            }],
            response_format={"type": "json_object"}
        )
        
        detection = json.loads(result.choices[0].message.content)
        return {
            'is_grounded': detection.get('is_grounded', True),
            'verdict': detection.get('verdict', 'GROUNDED'),
            'tokens': {
                'input': result.usage.prompt_tokens,
                'output': result.usage.completion_tokens
            }
        }

print("Agents defined successfully")

## MCP Servers (Simulated)

In [ ]:
class GitHubMCPServer:
    def __init__(self, client, token=None):
        self.client = client
        self.token = token
    
    def search_issues(self, query: str) -> dict:
        # Simulation: in production this would do a real GitHub search
        response = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": "Extract keywords for GitHub issue search. Return JSON with 'keywords' array."
            }, {
                "role": "user",
                "content": query
            }],
            response_format={"type": "json_object"}
        )
        
        # Second call for ranking
        ranking = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": "Rank simulated GitHub issues."
            }, {
                "role": "user",
                "content": f"Query: {query}"
            }]
        )
        
        return {
            'issues': [],
            'tokens': {
                'input': response.usage.prompt_tokens + ranking.usage.prompt_tokens,
                'output': response.usage.completion_tokens + ranking.usage.completion_tokens
            }
        }

class StackOverflowMCPServer:
    def __init__(self, client):
        self.client = client
    
    def search_posts(self, query: str) -> dict:
        # Simulation: keyword extraction
        keywords_resp = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": "Extract StackOverflow search keywords."
            }, {
                "role": "user",
                "content": query
            }]
        )
        
        # Filtering
        filter_resp = self.client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[{
                "role": "system",
                "content": "Filter StackOverflow results."
            }, {
                "role": "user",
                "content": query
            }]
        )
        
        return {
            'posts': [],
            'tokens': {
                'input': keywords_resp.usage.prompt_tokens + filter_resp.usage.prompt_tokens,
                'output': keywords_resp.usage.completion_tokens + filter_resp.usage.completion_tokens
            }
        }

print("MCP Servers defined successfully")

## Main AgenticRAG Class

In [ ]:
from datetime import datetime
import chromadb

class AgenticRAG:
    def __init__(self):
        self.openai_client = OpenAI()
        
        # Initialize ChromaDB
        self.chroma_client = chromadb.PersistentClient(path="./chroma_db")
        self.collection = self.chroma_client.get_collection(name="fabric_docs")
        
        # Initialize agents
        self.intent_classifier = IntentClassifier(self.openai_client)
        self.query_rewriter = QueryRewriter(self.openai_client)
        self.rag_scorer = RAGScorer(self.openai_client)
        self.context_synthesizer = ContextSynthesizer(self.openai_client)
        self.response_generator = ResponseGenerator(self.openai_client)
        self.code_validator = CodeValidator(self.openai_client)
        self.quality_evaluator = QualityEvaluator(self.openai_client)
        self.hallucination_detector = HallucinationDetector(self.openai_client)
        
        # Initialize MCP servers
        github_token = os.getenv('GITHUB_TOKEN')
        self.github_server = GitHubMCPServer(self.openai_client, github_token)
        self.stackoverflow_server = StackOverflowMCPServer(self.openai_client)
        
        self.total_tokens = {'input': 0, 'output': 0}
    
    def _embed_query(self, query: str) -> list:
        """Generate embedding for query"""
        response = self.openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=query
        )
        return response.data[0].embedding
    
    def _retrieve_chunks(self, query: str, top_k: int = 5) -> list:
        """Retrieve chunks from ChromaDB vector database"""
        query_embedding = self._embed_query(query)
        
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )
        
        chunks = []
        for i, (doc, metadata) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
            chunks.append({
                'text': doc,
                'doc_title': metadata.get('doc_title', 'Unknown'),
                'source': metadata.get('source', 'Unknown')
            })
        
        return chunks
    
    def _track_tokens(self, agent_result: dict):
        if 'tokens' in agent_result:
            self.total_tokens['input'] += agent_result['tokens'].get('input', 0)
            self.total_tokens['output'] += agent_result['tokens'].get('output', 0)
    
    def query(self, user_query: str, verbose: bool = True) -> dict:
        if verbose:
            print(f"\n{'='*70}")
            print(f"AGENTIC RAG QUERY")
            print(f"{'='*70}")
            print(f"Query: {user_query}\n")
        
        start_time = datetime.now()
        self.total_tokens = {'input': 0, 'output': 0}
        
        # Agent 1: Intent Classification
        if verbose: print("Agent 1: Intent Classification...")
        intent_result = self.intent_classifier.classify(user_query)
        self._track_tokens(intent_result)
        if verbose: print(f"   Category: {intent_result['category']}\n")
        
        # Agent 2: Query Rewriting
        if verbose: print("Agent 2: Query Rewriting...")
        rewrite_result = self.query_rewriter.rewrite(user_query, intent_result['category'])
        self._track_tokens(rewrite_result)
        if verbose: print(f"   Rewritten: {rewrite_result['rewritten_query'][:60]}...\n")
        
        # Real RAG retrieval from ChromaDB
        if verbose: print("Retrieving documents from ChromaDB...")
        raw_chunks = self._retrieve_chunks(rewrite_result['rewritten_query'], top_k=5)
        
        # Agent 3: RAG Scoring
        if verbose: print("Agent 3: Scoring & Ranking...")
        scorer_result = self.rag_scorer.score_chunks(user_query, raw_chunks)
        self._track_tokens(scorer_result)
        ranked_chunks = scorer_result['ranked_chunks'][:3]
        if verbose: print(f"   Top 3 chunks selected\n")
        
        # Agent 4: GitHub MCP
        if verbose: print("Agent 4: GitHub MCP Server...")
        github_result = self.github_server.search_issues(user_query)
        self._track_tokens(github_result)
        if verbose: print(f"   Searched GitHub\n")
        
        # Agent 5: StackOverflow MCP
        if verbose: print("Agent 5: StackOverflow MCP Server...")
        so_result = self.stackoverflow_server.search_posts(user_query)
        self._track_tokens(so_result)
        if verbose: print(f"   Searched StackOverflow\n")
        
        # Agent 6: Context Synthesis
        if verbose: print("Agent 6: Context Synthesis...")
        synthesis_result = self.context_synthesizer.synthesize(
            user_query, ranked_chunks, github_result['issues'], so_result['posts']
        )
        self._track_tokens(synthesis_result)
        synthesized_context = synthesis_result['synthesized_context']
        if verbose: print(f"   Context synthesized\n")
        
        # Agent 7: Response Generation
        if verbose: print("Agent 7: Response Generation...")
        response_result = self.response_generator.generate(
            user_query, synthesized_context, intent_result['category']
        )
        self._track_tokens(response_result)
        final_response = response_result['answer']
        if verbose: print(f"   Response generated\n")
        
        # Agent 8: Code Validation (if applicable)
        validation_result = None
        if response_result['has_code']:
            if verbose: print("Agent 8: Code Validation...")
            validation_result = self.code_validator.validate(final_response, synthesized_context)
            self._track_tokens(validation_result)
            if verbose: print(f"   Code validated\n")
        else:
            if verbose: print("Agent 8: Skipped (no code)\n")
        
        # Agent 9: Quality Evaluation
        if verbose: print("Agent 9: Quality Evaluation...")
        quality_result = self.quality_evaluator.evaluate(user_query, final_response)
        self._track_tokens(quality_result)
        if verbose: print(f"   Quality: {quality_result['scores']['overall']}/10\n")
        
        # Agent 10: Hallucination Detection
        if verbose: print("Agent 10: Hallucination Detection...")
        hallucination_result = self.hallucination_detector.detect(final_response, synthesized_context)
        self._track_tokens(hallucination_result)
        if verbose: print(f"   Verdict: {hallucination_result['verdict']}\n")
        
        # Calculate metrics
        elapsed = (datetime.now() - start_time).total_seconds()
        input_tokens = self.total_tokens['input']
        output_tokens = self.total_tokens['output']
        cost = (input_tokens * 0.01 / 1000) + (output_tokens * 0.03 / 1000)
        
        llm_calls = 10 + (2 if response_result['has_code'] else 0)
        
        if verbose:
            print(f"{'='*70}")
            print(f"SUMMARY")
            print(f"{'='*70}")
            print(f"LLM calls:        {llm_calls}")
            print(f"Input tokens:     {input_tokens:,}")
            print(f"Output tokens:    {output_tokens:,}")
            print(f"Total cost:       ${cost:.5f}")
            print(f"Latency:          {elapsed:.2f}s")
            print(f"Quality score:    {quality_result['scores']['overall']}/10")
            print(f"{'='*70}\n")
        
        return {
            'query': user_query,
            'response': final_response,
            'intent': intent_result['category'],
            'llm_calls': llm_calls,
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'cost': cost,
            'latency': elapsed,
            'quality_score': quality_result['scores']['overall'],
            'is_grounded': hallucination_result['is_grounded']
        }

print("AgenticRAG class defined successfully")

## Test with Multiple Queries

In [ ]:
def run_tests(queries: list):
    """Run tests on multiple queries and generate JSON statistics"""
    
    agent = AgenticRAG()
    results = []
    
    print(f"\n{'AGENTIC RAG TEST':^70}")
    print(f"{'='*70}")
    print(f"Test queries: {len(queries)}")
    print(f"{'='*70}\n")
    
    for i, query in enumerate(queries, 1):
        print(f"\n{'─'*70}")
        print(f"TEST {i}/{len(queries)}")
        print(f"{'─'*70}\n")
        
        result = agent.query(query, verbose=True)
        results.append(result)
    
    # Calculate aggregate statistics
    total_cost = sum(r['cost'] for r in results)
    avg_cost = total_cost / len(results)
    total_calls = sum(r['llm_calls'] for r in results)
    avg_calls = total_calls / len(results)
    avg_latency = sum(r['latency'] for r in results) / len(results)
    avg_quality = sum(r['quality_score'] for r in results) / len(results)
    total_input = sum(r['input_tokens'] for r in results)
    total_output = sum(r['output_tokens'] for r in results)
    
    summary = {
        'test_date': datetime.now().isoformat(),
        'model': 'gpt-4-turbo-preview',
        'architecture': 'agentic_rag_mcp',
        'queries': results,
        'summary': {
            'total_queries': len(results),
            'total_cost': round(total_cost, 5),
            'avg_cost_per_query': round(avg_cost, 5),
            'total_llm_calls': total_calls,
            'avg_llm_calls': round(avg_calls, 1),
            'avg_latency': round(avg_latency, 6),
            'avg_quality_score': round(avg_quality, 1),
            'total_input_tokens': total_input,
            'total_output_tokens': total_output
        }
    }
    
    # Print final report
    print(f"\n{'='*70}")
    print(f"{'FINAL REPORT':^70}")
    print(f"{'='*70}")
    print(f"Total queries:        {len(results)}")
    print(f"Total LLM calls:      {total_calls}")
    print(f"Avg calls/query:      {avg_calls:.1f}")
    print(f"Total cost:           ${total_cost:.2f}")
    print(f"Avg cost/query:       ${avg_cost:.5f}")
    print(f"Avg latency:          {avg_latency:.2f}s")
    print(f"Avg quality:          {avg_quality:.1f}/10")
    print(f"Total input tokens:   {total_input:,}")
    print(f"Total output tokens:  {total_output:,}")
    print(f"{'='*70}\n")
    
    return summary

print("Test function defined")

## Run Tests

In [ ]:
# Define test queries
test_queries = [
    "What is Hyperledger Fabric?",
    # Add more queries here if you want to test with multiple questions
]

# Run tests
summary = run_tests(test_queries)

## Display JSON Results

In [ ]:
import json

# Display only the summary in JSON format
print("\n" + "="*70)
print("JSON SUMMARY")
print("="*70 + "\n")
print(json.dumps(summary['summary'], indent=2))

# Save complete file
output_filename = f"agentic_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"\nComplete results saved to: {output_filename}")

# Download file (optional in Colab)
from google.colab import files
files.download(output_filename)

## Expected Output Example

```json
{
  "summary": {
    "total_queries": 1,
    "total_cost": 0.11868,
    "avg_cost_per_query": 0.11868,
    "total_llm_calls": 12,
    "avg_llm_calls": 12.0,
    "avg_latency": 57.992819,
    "avg_quality_score": 9.0,
    "total_input_tokens": 5742,
    "total_output_tokens": 2042
  }
}
```